In [2]:
import os
import pandas as pd
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.llms import Ollama
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [3]:
PDF_PATH = "eg_asp.pdf"#"IPA1.pdf"
LLM_MODEL = "llama3.1:latest"
EMBEDDING_MODEL = "mxbai-embed-large"

In [5]:


def setup_rag_chain(pdf_path: str):
    """
    Sets up the RAG chain by loading, chunking, and embedding the PDF,
    then creating a retriever and the final LLM chain.
    """
    print("Step 1: Loading the PDF document...")
    # Load the PDF document
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    # Split the document into chunks
    print("Step 2: Splitting the document into chunks...")
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    docs = text_splitter.split_documents(documents)
    
    # Check if there are any documents to process
    if not docs:
        print("Error: The PDF file is empty or could not be processed.")
        return None

    # Create the embeddings model
    print("Step 3: Creating embeddings...")
    embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)

    # Store the chunks and embeddings in ChromaDB
    print("Step 4: Creating a ChromaDB vector store...")
    vector_store = Chroma.from_documents(documents=docs, embedding=embeddings)
    
    # Create a retriever
    retriever = vector_store.as_retriever(search_kwargs={"k": 5})

    # Set up the LLM
    print("Step 5: Setting up the Ollama LLM...")
    llm = Ollama(model=LLM_MODEL)

    # Create the prompt template
    prompt = ChatPromptTemplate.from_template("""
    You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question.
    If you don't know the answer, just say that you don't know. Use a polite and conversational tone.
    
    Question: {question}
    Context: {context}
    Answer:
    """)

    # Define the RAG chain
    print("Step 6: Assembling the RAG chain...")
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    return rag_chain

def chat_with_pdf():
    """
    Main function to run the PDF chatbot.
    """
    rag_chain = setup_rag_chain(PDF_PATH)
    
    if rag_chain is None:
        return

    print("\n-------------------------------------------")
    print(f"Chatbot is ready. You can now ask questions about '{os.path.basename(PDF_PATH)}'.")
    print("Type 'exit' to quit.")
    print("-------------------------------------------")
    
    while True:
        user_question = input("\nYour question: ")
        if user_question.lower() == 'exit':
            print("Goodbye!")
            break
        
        print("Thinking...")
        try:
            response = rag_chain.invoke(user_question)
            print(f"\nChatbot: {response}")
        except Exception as e:
            print(f"An error occurred: {e}")
            print("Please make sure the Ollama server is running and the models are pulled.")
            break

if __name__ == "__main__":
    # Before running this script:
    # 1. Make sure you have Ollama installed and running.
    # 2. Pull the required models by running 'ollama pull llama3' and 'ollama pull nomic-embed-text'
    #    in your terminal.
    # 3. Place your PDF file in the same directory as this script and rename it to 'your_document.pdf'.
    
    # Start the chatbot
    chat_with_pdf()


Step 1: Loading the PDF document...
Step 2: Splitting the document into chunks...
Step 3: Creating embeddings...
Step 4: Creating a ChromaDB vector store...
Step 5: Setting up the Ollama LLM...
Step 6: Assembling the RAG chain...

-------------------------------------------
Chatbot is ready. You can now ask questions about 'eg_asp.pdf'.
Type 'exit' to quit.
-------------------------------------------



Your question:  explain the whole process in simple steps


Thinking...

Chatbot: Let me break down the process in simple steps for you:

**Step 1: Feed Preparation**
The feed is pumped out of a storage tank and the pressure is raised from 1 atm to 13 atm.

**Step 2: Heating**
The stream (Stream 1, S1) is heated up from 35°C to 190°C to meet the desired conditions for producing ethylene glycol.

**Step 3: Reaction in the Reactor**
The heated stream (S2) is fed into a reactor where reactions take place. After the reactions are completed, both liquid products and vapors are produced.

**Step 4: Mixing**
Since the vapors and liquid products contain the same constituents, a mixer is used to ensure that all components of the products are present in the produced stream (labeled as "product").

**Step 5: Pressure Reduction**
A control valve is used to reduce the pressure of the stream before it's sent to an atmospheric distillation unit.

**Step 6: Distillation**
The distillation process separates the product into EG (ethylene glycol), DEG (diethylene


Your question:  input chemical composition


Thinking...

Chatbot: To determine the input chemical composition for the distillation process, I'll look at Table 6 and Table 10 in the provided context.

Table 6: Composition of Vapor product.
H2O: 311.228 kg/h
EG: 0.0000 kg/h
DEG: 0.0000 kg/h

Table 10: Composition of Distillates (D1).
H2O: 0.0000 kg/h
EG: 38.5854 kg/h
DEG: 0.0015 kg/h

Since the question asks for the input chemical composition, I will assume it refers to the feed or initial composition of the mixture. Based on the tables, the input composition appears to be dominated by H2O and EG.

If we consider the Mass Flow [kg/h] values for each component:

* H2O: 311.228 kg/h (Table 6)
* EG: 38.5854 kg/h (Table 10)

However, it's essential to note that these tables show different components and their masses in various parts of the process. Without additional context or information about which stream is considered the input for the distillation process, I can only provide a partial answer.

If you could specify which stream or

...

Thinking...

Chatbot: It seems like you're looking for information about the feed in a process related to producing ethylene glycol (EG). Based on the context provided, here's what I can tell you:

* The composition of the feed is: EO (ethylene oxide) = 27.5 kg/h and water = 550 kg/h.
* The total amount of effluents produced in 25 days is approximately 3465 kg.
* In one cycle/day to produce EG, it takes 6 hours, resulting in a feed rate of 577.5 kg/h.
* A UNIQUAC Ideal fluid package was chosen for this process.

Let me know if you'd like more specific information or details about the feed!



Your question:  write a stage wise data for the process include input feed temperature and pressure and chemical composition , next equipment and what is happening in it and similarly every stage of the process


Thinking...

Chatbot: Here is the stage-wise data for the process:

**Stage 1: Storage Tank**

* Input:
	+ Feed temperature: 35°C
	+ Feed pressure: 1 atm
	+ Chemical composition:
		- EO (Ethylene Oxide): 27.5 kg/h
		- Water: 550 kg/h
		- Ethylene Glycol (EG): 577.5 kg/h
		- Di-Ethylene Glycol (DEG): 577.5 kg/h
* Next equipment: Pump

**Stage 2: Pump**

* Input:
	+ Stream S1 from storage tank
* Process: Raise pressure to 13 atm
* Output:
	+ Stream S1 with increased pressure
	+ Pressure: 13 atm

**Stage 3: Heater**

* Input:
	+ Stream S1 from pump
* Process: Heat up stream S1 to 190°C
* Output:
	+ Stream S2 at 190°C and 1317 kPa
	+ Temperature: 190°C
	+ Pressure: 1317 kPa

**Stage 4: Reactor**

* Input:
	+ Stream S2 from heater
* Process: Chemical reactions take place
* Output:
	+ Vapors containing minor amount of EG, DEG, and steam
	+ Liquid products comprising bulk of EG, minor amount of DEG, and huge amount of water

**Stage 5: Mixer**

* Input:
	+ Vapors and liquid products from reac


Your question:  exit


Goodbye!


In [8]:
import math
from enum import Enum

# --- Simplified Placeholder Classes (Essential for calculation logic) ---

class StreamSpec(Enum):
    """Defines how a stream's state is specified."""
    TEMPERATURE_AND_PRESSURE = 0
    PRESSURE_AND_ENTHALPY = 1
    PRESSURE_AND_VAPOR_FRACTION = 2

class FlashCalculationType(Enum):
    """Types of flash calculations needed for property determination."""
    PRESSURE_ENTHALPY = 0  # Calculate T, VF from P, H
    PRESSURE_TEMPERATURE = 1 # Calculate H, VF from P, T
    PRESSURE_VAPOR_FRACTION = 2 # Calculate T, H from P, VF

class StreamProperties:
    """Holds the thermodynamic properties of a stream."""
    def __init__(self, temperature=None, pressure=None, enthalpy=None, mass_flow=None, vapor_fraction=None):
        self.temperature = temperature  # Kelvin (K)
        self.pressure = pressure      # Pascals (Pa)
        self.enthalpy = enthalpy      # kJ/kg
        self.mass_flow = mass_flow    # kg/s
        self.vapor_fraction = vapor_fraction # Molar vapor fraction (0 to 1)

class MaterialStream:
    """Represents a material stream with its properties."""
    def __init__(self, name=""):
        self.name = name
        self.properties = StreamProperties()
        self.spec_type = None # How the stream was specified/calculated

    def __str__(self):
        # Handle None values for formatting
        temp_str = f"{self.properties.temperature:.2f}" if self.properties.temperature is not None else "N/A"
        pressure_str = f"{self.properties.pressure:.2f}" if self.properties.pressure is not None else "N/A"
        enthalpy_str = f"{self.properties.enthalpy:.2f}" if self.properties.enthalpy is not None else "N/A"
        mass_flow_str = f"{self.properties.mass_flow:.2f}" if self.properties.mass_flow is not None else "N/A"
        vapor_fraction_str = f"{self.properties.vapor_fraction:.2f}" if self.properties.vapor_fraction is not None else "N/A"

        return (f"Stream '{self.name}':\n"
                f"  T: {temp_str} K\n"
                f"  P: {pressure_str} Pa\n"
                f"  H: {enthalpy_str} kJ/kg\n"
                f"  W: {mass_flow_str} kg/s\n"
                f"  VF: {vapor_fraction_str}")

class FlashResult:
    """Result object for a flash calculation."""
    def __init__(self, temp=None, enthalpy=None, vapor_fraction=None, pressure=None):
        self.calculated_temperature = temp
        self.calculated_enthalpy = enthalpy
        self.calculated_vapor_fraction = vapor_fraction
        self.calculated_pressure = pressure

class PropertyPackage:
    """
    A *highly simplified* placeholder for a thermodynamic property package.
    In a real simulator, this would be a sophisticated library for phase equilibrium
    and property calculations (e.g., using equations of state like Peng-Robinson).
    """
    def calculate_equilibrium(self, flash_type, val1, val2, initial_guess=None):
        """
        Simulates basic flash calculations. These are arbitrary relationships
        for demonstration purposes only, not based on real thermodynamics.
        """
        if flash_type == FlashCalculationType.PRESSURE_ENTHALPY:
            # val1 = Pressure, val2 = Enthalpy
            temp = 298.15 + (val2 / 1000) * 0.5 # Higher H -> Higher T
            vf = min(1.0, max(0.0, (val2 - 100) / 500)) # Higher H -> More vapor
            return FlashResult(temp=temp, enthalpy=val2, vapor_fraction=vf, pressure=val1)
        
        elif flash_type == FlashCalculationType.PRESSURE_TEMPERATURE:
            # val1 = Pressure, val2 = Temperature
            enthalpy = 100 + (val2 - 298.15) * 2 # Higher T -> Higher H
            vf = min(1.0, max(0.0, (val2 - 350) / 100)) # Higher T -> More vapor
            return FlashResult(temp=val2, enthalpy=enthalpy, vapor_fraction=vf, pressure=val1)
        
        elif flash_type == FlashCalculationType.PRESSURE_VAPOR_FRACTION:
            # val1 = Pressure, val2 = Vapor Fraction
            enthalpy = 100 + val2 * 500 # Higher VF -> Higher H
            temp = 298.15 + val2 * 100 # Higher VF -> Higher T
            return FlashResult(temp=temp, enthalpy=enthalpy, vapor_fraction=val2, pressure=val1)
        
        return FlashResult() # Default empty result

# --- End Simplified Placeholder Classes ---

class HeaterCalculation:
    """
    A simplified class focusing on the core calculation logic of a heater.
    It takes an inlet stream and calculates an outlet stream based on a
    specified calculation mode.
    """
    class CalculationMode(Enum):
        """Defines how the heater operation is specified."""
        HEAT_ADDED = 0
        OUTLET_TEMPERATURE = 1
        OUTLET_VAPOR_FRACTION = 3 # Skipping EnergyStream and TemperatureChange for maximum simplification

    def __init__(self, name="Heater"):
        self.name = name
        self.inlet_stream = None
        self.outlet_stream = MaterialStream(name=f"{name} Outlet")
        self.property_package = PropertyPackage()

        # Configurable properties for calculation
        self.calc_mode = self.CalculationMode.HEAT_ADDED
        self.heat_duty = None       # Q (kW), used in HEAT_ADDED mode
        self.outlet_temp_setpoint = None # T2 (K), used in OUTLET_TEMPERATURE mode
        self.outlet_vf_setpoint = None  # VF2 (fraction), used in OUTLET_VAPOR_FRACTION mode
        self.pressure_drop = 0.0    # Delta P (Pa)
        self.efficiency = 100.0     # Efficiency (%)

        # Calculated properties (results)
        self.calculated_delta_q = None # Q (kW)
        self.calculated_delta_t = None # Delta T (K)
        self.calculated_outlet_temp = None # T2 (K)
        self.calculated_outlet_vf = None # VF2 (fraction)

    def set_inlet_stream(self, stream: MaterialStream):
        """Sets the inlet stream for the heater calculation."""
        if not isinstance(stream, MaterialStream):
            raise TypeError("Inlet must be a MaterialStream object.")
        if any(prop is None for prop in [stream.properties.temperature, 
                                         stream.properties.pressure, 
                                         stream.properties.enthalpy, 
                                         stream.properties.mass_flow]):
            raise ValueError("Inlet stream properties (T, P, H, Mass Flow) must be defined.")
        self.inlet_stream = stream

    def calculate(self):
        """
        Performs the heater calculation based on the selected mode.
        """
        if self.inlet_stream is None:
            raise ValueError("Inlet stream is not set. Cannot perform calculation.")

        # Get inlet stream properties
        ti = self.inlet_stream.properties.temperature
        pi = self.inlet_stream.properties.pressure
        hi = self.inlet_stream.properties.enthalpy
        wi = self.inlet_stream.properties.mass_flow

        # Calculate outlet pressure
        p2 = pi - self.pressure_drop
        if p2 <= 0:
            raise ValueError(f"Outlet pressure calculated as {p2} Pa, which is non-positive. Check pressure drop.")

        q_eff = 0.0 # Effective heat added to the fluid
        t2 = None
        h2 = None
        vf2 = None

        if self.calc_mode == self.CalculationMode.HEAT_ADDED:
            if self.heat_duty is None:
                raise ValueError("Heat duty (heat_duty) must be specified for HEAT_ADDED mode.")
            
            # Effective heat added (kW)
            q_eff = self.heat_duty * (self.efficiency / 100.0) 
            
            # Enthalpy balance: H_out = H_in + Q_eff / MassFlow
            # Ensure mass_flow is not zero to avoid division by zero
            if wi == 0:
                raise ValueError("Inlet mass flow cannot be zero in HEAT_ADDED mode.")
            h2 = hi + q_eff / wi
            
            # Perform PH Flash to get T2 and VF2
            flash_result = self.property_package.calculate_equilibrium(FlashCalculationType.PRESSURE_ENTHALPY, p2, h2)
            t2 = flash_result.calculated_temperature
            vf2 = flash_result.calculated_vapor_fraction
            
            self.calculated_delta_q = self.heat_duty

        elif self.calc_mode == self.CalculationMode.OUTLET_TEMPERATURE:
            if self.outlet_temp_setpoint is None:
                raise ValueError("Outlet temperature setpoint (outlet_temp_setpoint) must be specified for OUTLET_TEMPERATURE mode.")
            
            t2 = self.outlet_temp_setpoint
            
            # Perform PT Flash to get H2 and VF2
            flash_result = self.property_package.calculate_equilibrium(FlashCalculationType.PRESSURE_TEMPERATURE, p2, t2)
            h2 = flash_result.calculated_enthalpy
            vf2 = flash_result.calculated_vapor_fraction

            # Calculate required heat duty: Q = (H_out - H_in) * MassFlow / (Efficiency / 100)
            if self.efficiency == 0:
                 raise ValueError("Efficiency cannot be zero when calculating heat duty.")
            q_required = (h2 - hi) * wi / (self.efficiency / 100.0)
            self.calculated_delta_q = q_required

        elif self.calc_mode == self.CalculationMode.OUTLET_VAPOR_FRACTION:
            if self.outlet_vf_setpoint is None:
                raise ValueError("Outlet vapor fraction setpoint (outlet_vf_setpoint) must be specified for OUTLET_VAPOR_FRACTION mode.")
            
            vf2 = self.outlet_vf_setpoint
            
            # Perform PVF Flash to get T2 and H2
            flash_result = self.property_package.calculate_equilibrium(FlashCalculationType.PRESSURE_VAPOR_FRACTION, p2, vf2)
            t2 = flash_result.calculated_temperature
            h2 = flash_result.calculated_enthalpy

            # Calculate required heat duty
            if self.efficiency == 0:
                 raise ValueError("Efficiency cannot be zero when calculating heat duty.")
            q_required = (h2 - hi) * wi / (self.efficiency / 100.0)
            self.calculated_delta_q = q_required
        
        else:
            raise ValueError(f"Unsupported calculation mode: {self.calc_mode.name}")

        # Update calculated properties of the HeaterCalculation object
        self.calculated_outlet_temp = t2
        self.calculated_outlet_vf = vf2
        self.calculated_delta_t = t2 - ti if t2 is not None else None

        # Assign calculated values to the outlet stream
        self.outlet_stream.properties.temperature = t2
        self.outlet_stream.properties.pressure = p2
        self.outlet_stream.properties.enthalpy = h2
        self.outlet_stream.properties.mass_flow = wi # Mass flow is conserved
        self.outlet_stream.properties.vapor_fraction = vf2
        
        # Set the specification type for the outlet stream
        if self.calc_mode == self.CalculationMode.HEAT_ADDED:
            self.outlet_stream.spec_type = StreamSpec.PRESSURE_AND_ENTHALPY
        elif self.calc_mode in [self.CalculationMode.OUTLET_TEMPERATURE, self.CalculationMode.OUTLET_VAPOR_FRACTION]:
            self.outlet_stream.spec_type = StreamSpec.TEMPERATURE_AND_PRESSURE


# --- Example Usage ---

if __name__ == "__main__":
    # 1. Create an Inlet Stream
    inlet = MaterialStream(name="Feed Stream")
    inlet.properties.temperature = 300.0  # K
    inlet.properties.pressure = 101325.0 # Pa (1 atm)
    inlet.properties.enthalpy = 50.0     # kJ/kg
    inlet.properties.mass_flow = 1.0     # kg/s

    print("--- Heater Calculation Examples ---")
    print("\nInitial Inlet Stream:")
    print(inlet)

    # --- Mode 1: Heat Added ---
    heater_q = HeaterCalculation(name="Heater_Q")
    heater_q.set_inlet_stream(inlet)
    heater_q.calc_mode = HeaterCalculation.CalculationMode.HEAT_ADDED
    heater_q.heat_duty = 500.0  # Add 500 kW of heat
    heater_q.pressure_drop = 1000.0 # 1 kPa pressure drop
    heater_q.efficiency = 90.0 # 90% efficient

    try:
        heater_q.calculate()
        print("\n--- HEAT ADDED Mode ---")
        print(f"  Specified Heat Duty: {heater_q.heat_duty} kW")
        print(f"  Calculated Delta Q: {heater_q.calculated_delta_q:.2f} kW")
        print(f"  Calculated Delta T: {heater_q.calculated_delta_t:.2f} K")
        print("  Outlet Stream:")
        print(heater_q.outlet_stream)
    except Exception as e:
        print(f"Error in HEAT ADDED mode: {e}")

    # --- Mode 2: Outlet Temperature ---
    heater_t = HeaterCalculation(name="Heater_T")
    heater_t.set_inlet_stream(inlet)
    heater_t.calc_mode = HeaterCalculation.CalculationMode.OUTLET_TEMPERATURE
    heater_t.outlet_temp_setpoint = 450.0 # Target 450 K at outlet
    heater_t.pressure_drop = 500.0 # 0.5 kPa pressure drop
    heater_t.efficiency = 85.0 # 85% efficient

    try:
        heater_t.calculate()
        print("\n--- OUTLET TEMPERATURE Mode ---")
        print(f"  Specified Outlet Temp: {heater_t.outlet_temp_setpoint} K")
        print(f"  Calculated Heat Duty: {heater_t.calculated_delta_q:.2f} kW")
        print(f"  Calculated Delta T: {heater_t.calculated_delta_t:.2f} K")
        print("  Outlet Stream:")
        print(heater_t.outlet_stream)
    except Exception as e:
        print(f"Error in OUTLET TEMPERATURE mode: {e}")

    # --- Mode 3: Outlet Vapor Fraction ---
    heater_vf = HeaterCalculation(name="Heater_VF")
    heater_vf.set_inlet_stream(inlet)
    heater_vf.calc_mode = HeaterCalculation.CalculationMode.OUTLET_VAPOR_FRACTION
    heater_vf.outlet_vf_setpoint = 0.8 # Target 80% vapor fraction at outlet
    heater_vf.pressure_drop = 750.0 # 0.75 kPa pressure drop
    heater_vf.efficiency = 92.0 # 92% efficient

    try:
        heater_vf.calculate()
        print("\n--- OUTLET VAPOR FRACTION Mode ---")
        print(f"  Specified Outlet VF: {heater_vf.outlet_vf_setpoint}")
        print(f"  Calculated Heat Duty: {heater_vf.calculated_delta_q:.2f} kW")
        print(f"  Calculated Delta T: {heater_vf.calculated_delta_t:.2f} K")
        print("  Outlet Stream:")
        print(heater_vf.outlet_stream)
    except Exception as e:
        print(f"Error in OUTLET VAPOR FRACTION mode: {e}")

--- Heater Calculation Examples ---

Initial Inlet Stream:
Stream 'Feed Stream':
  T: 300.00 K
  P: 101325.00 Pa
  H: 50.00 kJ/kg
  W: 1.00 kg/s
  VF: N/A

--- HEAT ADDED Mode ---
  Specified Heat Duty: 500.0 kW
  Calculated Delta Q: 500.00 kW
  Calculated Delta T: -1.60 K
  Outlet Stream:
Stream 'Heater_Q Outlet':
  T: 298.40 K
  P: 100325.00 Pa
  H: 500.00 kJ/kg
  W: 1.00 kg/s
  VF: 0.80

--- OUTLET TEMPERATURE Mode ---
  Specified Outlet Temp: 450.0 K
  Calculated Heat Duty: 416.12 kW
  Calculated Delta T: 150.00 K
  Outlet Stream:
Stream 'Heater_T Outlet':
  T: 450.00 K
  P: 100825.00 Pa
  H: 403.70 kJ/kg
  W: 1.00 kg/s
  VF: 1.00

--- OUTLET VAPOR FRACTION Mode ---
  Specified Outlet VF: 0.8
  Calculated Heat Duty: 489.13 kW
  Calculated Delta T: 78.15 K
  Outlet Stream:
Stream 'Heater_VF Outlet':
  T: 378.15 K
  P: 100575.00 Pa
  H: 500.00 kJ/kg
  W: 1.00 kg/s
  VF: 0.80


In [ ]:
import math

class MaterialStream:
    """
    A simplified representation of a material stream for cooler calculations.
    In a real simulation, this would be much more complex.
    """
    def __init__(self, temperature=None, pressure=None, enthalpy=None, mass_flow=None, components=None):
        self.temperature = temperature  # K
        self.pressure = pressure      # Pa
        self.enthalpy = enthalpy      # kJ/kg
        self.mass_flow = mass_flow    # kg/s
        self.components = components if components is not None else {} # {component_name: mole_fraction}
        self.vapor_fraction = None    # Dimensionless

    def get_temperature(self):
        return self.temperature

    def get_pressure(self):
        return self.pressure

    def get_mass_enthalpy(self):
        return self.enthalpy

    def get_mass_flow(self):
        return self.mass_flow

    def set_temperature(self, temp):
        self.temperature = temp

    def set_pressure(self, press):
        self.pressure = press

    def set_mass_enthalpy(self, enth):
        self.enthalpy = enth

    def set_vapor_phase_mole_fraction(self, vf):
        self.vapor_fraction = vf

    def assign_from_phase(self, phase_label, source_stream, copy_components=False):
        # Simplified: copy properties from the source stream's mixture
        self.temperature = source_stream.temperature
        self.pressure = source_stream.pressure
        self.enthalpy = source_stream.enthalpy
        self.mass_flow = source_stream.mass_flow
        self.vapor_fraction = source_stream.vapor_fraction
        if copy_components:
            self.components = source_stream.components.copy()

class EnergyStream:
    """
    A simplified representation of an energy stream.
    """
    def __init__(self, energy_flow=None):
        self.energy_flow = energy_flow # kW

    def get_energy_flow(self):
        return self.energy_flow

    def set_energy_flow(self, flow):
        self.energy_flow = flow

class FlashCalculationResult:
    """
    A placeholder for the result of a flash calculation.
    """
    def __init__(self, calculated_temperature=None, calculated_enthalpy=None, vapor_phase_mole_fraction=None):
        self.calculated_temperature = calculated_temperature
        self.calculated_enthalpy = calculated_enthalpy
        self.vapor_phase_mole_fraction = vapor_phase_mole_fraction

    def get_vapor_phase_mole_fraction(self):
        return self.vapor_phase_mole_fraction

class PropertyPackage:
    """
    A placeholder for a thermodynamic property package.
    In a real application, this would use a library like CoolProp.
    """
    def __init__(self):
        self.current_material_stream = None

    def calculate_equilibrium2(self, flash_type, p_or_vf, h_or_t, initial_guess=None):
        """
        Simulates a flash calculation.
        This is a *highly simplified* placeholder.
        Real flash calculations are complex and require a thermodynamic model.

        :param flash_type: "PressureEnthalpy", "PressureTemperature", "PressureVaporFraction", "VolumeTemperature"
        :param p_or_vf: Pressure (Pa) or Vapor Fraction (dimensionless)
        :param h_or_t: Enthalpy (kJ/kg) or Temperature (K)
        :param initial_guess: An initial guess for the iteration (e.g., for T in PH flash).
        :return: FlashCalculationResult object
        """
        # This part would contain the actual thermodynamic model logic.
        # For demonstration, we'll just return some plausible values based on the input.
        if flash_type == "PressureEnthalpy":
            # Given P and H, calculate T and VF
            temp = h_or_t * 0.1 + 273.15  # Placeholder: Enthalpy to temp conversion
            vf = 0.5 if h_or_t > 500 else 0.0 # Placeholder: Simple phase rule
            return FlashCalculationResult(calculated_temperature=temp, calculated_enthalpy=h_or_t, vapor_phase_mole_fraction=vf)
        elif flash_type == "PressureTemperature":
            # Given P and T, calculate H and VF
            enthalpy = h_or_t * 10 - 273.15 * 10 # Placeholder: Temp to enthalpy conversion
            vf = 0.5 if h_or_t > 350 else 0.0 # Placeholder: Simple phase rule
            return FlashCalculationResult(calculated_temperature=h_or_t, calculated_enthalpy=enthalpy, vapor_phase_mole_fraction=vf)
        elif flash_type == "PressureVaporFraction":
            # Given P and VF, calculate T and H
            temp = p_or_vf * 0.001 + 300 if h_or_t == 0.0 else initial_guess # Placeholder
            enthalpy = temp * 10 - 273.15 * 10 # Placeholder: Temp to enthalpy conversion
            return FlashCalculationResult(calculated_temperature=temp, calculated_enthalpy=enthalpy, vapor_phase_mole_fraction=p_or_vf)
        elif flash_type == "VolumeTemperature":
            # Given V and T, calculate P (and maybe other properties)
            pressure = 101325 * (h_or_t / 298.15) * (initial_guess / p_or_vf) if initial_guess else 101325 # Placeholder
            return FlashCalculationResult(calculated_pressure=pressure) # Assuming we add calculated_pressure to FlashCalculationResult if needed for dynamic mode.
        else:
            raise ValueError(f"Unknown flash type: {flash_type}")

class Cooler:
    """
    Python equivalent of the Cooler class from DWSIM.
    This focuses on the 'Calculate' and 'RunDynamicModel' methods.
    """
    class CalculationMode:
        HeatRemoved = 0
        OutletTemperature = 1
        OutletVaporFraction = 2
        TemperatureChange = 3
        EnergyStream = 4

    def __init__(self, name="Cooler", description=""):
        self.name = name
        self.description = description
        self.delta_p = 0.0  # Pressure drop (Pa)
        self.delta_t = None # Temperature change (K)
        self.delta_q = None # Heat removed (kW)
        self.outlet_temperature = 298.15 # K
        self.outlet_vapor_fraction = None
        self.calc_mode = self.CalculationMode.HeatRemoved
        self.efficiency = 100.0 # %

        # Placeholder for connected streams and property package
        self.inlet_material_stream = None
        self.outlet_material_stream = None
        self.inlet_energy_stream = None # For EnergyStream calculation mode
        self.outlet_energy_stream = None
        self.property_package = PropertyPackage()
        self.debug_mode = False

        # Dynamic mode properties
        self.flow_conductance = 1.0
        self.volume = 1.0
        self.minimum_pressure = 101325.0
        self.initialize_using_inlet_stream = True
        self.reset_content = False
        self.accumulation_stream = None
        self.prev_m = 0.0
        self.current_m = 0.0

    def set_inlet_material_stream(self, stream):
        self.inlet_material_stream = stream

    def set_outlet_material_stream(self, stream):
        self.outlet_material_stream = stream

    def set_inlet_energy_stream(self, stream):
        self.inlet_energy_stream = stream

    def set_outlet_energy_stream(self, stream):
        self.outlet_energy_stream = stream

    def set_property_package(self, pkg):
        self.property_package = pkg

    def set_debug_mode(self, mode):
        self.debug_mode = mode

    def append_debug_line(self, message):
        if self.debug_mode:
            print(f"DEBUG: {message}")

    def check_spec(self, value, is_temperature, spec_name):
        # This would be a more robust validation in a real system
        if value is None or (is_temperature and value <= 0): # Temperatures should be positive in K
            raise Exception(f"Invalid {spec_name} calculated: {value}")

    def run_dynamic_model(self, timestep, integrator_should_calculate_equilibrium=True):
        """
        Simulates the dynamic behavior of the cooler.
        This is a simplified translation and assumes streams can be 'added' or 'subtracted'
        which would be complex material and energy balances in a real simulator.
        """
        ims = self.inlet_material_stream
        oms = self.outlet_material_stream

        if ims is None or oms is None:
            raise Exception("Inlet and outlet material streams must be connected for dynamic mode.")

        if self.reset_content:
            self.accumulation_stream = None
            self.reset_content = False

        if self.accumulation_stream is None:
            if self.initialize_using_inlet_stream and ims.temperature is not None and ims.pressure is not None and ims.mass_flow is not None:
                # Clone inlet stream and set its properties
                self.accumulation_stream = MaterialStream(
                    temperature=ims.temperature,
                    pressure=ims.pressure,
                    enthalpy=ims.enthalpy, # Assuming enthalpy is available from inlet
                    mass_flow=ims.mass_flow, # Initial mass flow
                    components=ims.components.copy()
                )
                # Set initial mass in the volume
                if self.accumulation_stream.enthalpy is not None and self.accumulation_stream.pressure is not None:
                    # Need to estimate density for initial volume fill
                    # This is a major simplification. Real density calc requires flash.
                    # For now, let's assume a dummy density if we can't get it from a flash.
                    try:
                        temp_calc = self.property_package.calculate_equilibrium2("PressureEnthalpy", self.accumulation_stream.pressure, self.accumulation_stream.enthalpy)
                        # Assume some way to get molar volume or density from the flash result
                        molar_volume = 0.025 # Placeholder for m3/mol
                        if self.accumulation_stream.mass_flow > 0:
                             # Assume molar mass for density calc
                            avg_molar_mass = 0.029 # kg/mol, approx air
                            density = avg_molar_mass / molar_volume # kg/m3
                            self.accumulation_stream.mass_flow = density * self.volume # Initial mass in the volume
                            self.accumulation_stream.set_mass_flow(density * self.volume)
                        else:
                             self.accumulation_stream.set_mass_flow(100.0) # Arbitrary initial mass
                    except Exception as e:
                        self.append_debug_line(f"Warning: Could not get density for initial accumulation stream. Using arbitrary mass flow: {e}")
                        self.accumulation_stream.set_mass_flow(100.0) # Arbitrary initial mass

                self.accumulation_stream.set_temperature(ims.get_temperature())
                self.accumulation_stream.set_pressure(ims.get_pressure())
                self.accumulation_stream.enthalpy = ims.get_mass_enthalpy() # Ensure enthalpy is set if temp/pressure are

            else:
                # If not initializing from inlet, start with an empty stream (simplified)
                self.accumulation_stream = MaterialStream(temperature=298.15, pressure=self.minimum_pressure, mass_flow=0.0)
                # Need a base state for components
                if ims.components:
                    self.accumulation_stream.components = ims.components.copy()
                else:
                    self.accumulation_stream.components = {"Water": 1.0} # Default if no inlet components

        # In/out material balance (simplified)
        in_mass_flow = ims.get_mass_flow() if ims.get_mass_flow() is not None else 0.0
        out_mass_flow = oms.get_mass_flow() if oms.get_mass_flow() is not None else 0.0

        # Accumulation calculation: Net change in mass in the vessel over timestep
        # This is a simplification. A proper dynamic model would track mass and energy balance within the volume.
        # Here, it seems to be updating the accumulation stream itself based on inlet/outlet flows.
        # The original VB code's "Subtract(oms, timestep)" seems to imply changing the *accumulation stream's* properties
        # based on outlet flow, which is non-standard for accumulation.
        # We will interpret it as updating the mass and then recalculating properties.

        # Calculate accumulation stream properties based on current state and heat duty
        ha = self.accumulation_stream.get_mass_enthalpy() if self.accumulation_stream.get_mass_enthalpy() is not None else 0.0
        wa = self.accumulation_stream.get_mass_flow() if self.accumulation_stream.get_mass_flow() is not None else 0.0

        q_val = self.delta_q if self.delta_q is not None else 0.0 # Heat removed from the system (negative for cooling)

        if q_val != 0.0:
            if wa > 0:
                # Apply heat removal to the accumulated mass
                new_enthalpy = ha + q_val * timestep / wa
                self.accumulation_stream.set_mass_enthalpy(new_enthalpy)
                self.property_package.current_material_stream = self.accumulation_stream
                if integrator_should_calculate_equilibrium:
                    # Calculate new temperature based on updated enthalpy and current pressure
                    current_pressure = self.accumulation_stream.get_pressure()
                    if current_pressure is None:
                        current_pressure = self.minimum_pressure
                    flash_result = self.property_package.calculate_equilibrium2(
                        "PressureEnthalpy", current_pressure, self.accumulation_stream.get_mass_enthalpy()
                    )
                    self.accumulation_stream.set_temperature(flash_result.calculated_temperature)
            else:
                self.append_debug_line("Warning: Accumulation stream mass flow is zero, cannot apply heat duty dynamically.")

        # Calculate pressure (simplified, based on molar volume and temperature)
        molar_flow_acc = self.accumulation_stream.mass_flow / self.get_average_molar_mass(self.accumulation_stream.components) if self.accumulation_stream.mass_flow > 0 else 0.0
        
        # This part of the VB code for pressure calculation is quite specific to DWSIM's internal handling
        # of molar volume and equilibrium flashes for dynamic pressure.
        # `currentM = Vol / M` where M is molar flow, meaning currentM is molar volume (m3/mol).
        # `Pressure = currentM / prevM * Pressure` implies an ideal gas like pressure change with volume.

        if molar_flow_acc > 0:
            self.prev_m = self.current_m
            self.current_m = self.volume / molar_flow_acc

            self.property_package.current_material_stream = self.accumulation_stream
            current_pressure_acc = self.accumulation_stream.get_pressure()
            if current_pressure_acc is None:
                current_pressure_acc = self.minimum_pressure

            if current_pressure_acc > 0:
                if self.prev_m == 0.0 or integrator_should_calculate_equilibrium:
                    # Perform a volume-temperature flash to calculate pressure
                    # This needs a PropertyPackage that supports VolumeTemperature flash
                    flash_result = self.property_package.calculate_equilibrium2(
                        "VolumeTemperature", self.current_m, self.accumulation_stream.get_temperature(), current_pressure_acc
                    )
                    pressure_calc = flash_result.calculated_pressure
                else:
                    # Simplified pressure calculation based on volume ratio (like ideal gas law for constant T, n)
                    pressure_calc = (self.current_m / self.prev_m) * current_pressure_acc
            else:
                pressure_calc = self.minimum_pressure
        else:
            pressure_calc = self.minimum_pressure

        self.accumulation_stream.set_pressure(pressure_calc)

        # Update outlet stream based on accumulation stream (simplified)
        # In dynamic mode, the outlet stream properties are often determined by the current state
        # of the unit operation's internal accumulation.
        # The original code's DeltaP calculation (Wi / Kr)^2 implies a pressure drop based on flow and conductance.
        # This pressure drop is then subtracted from the accumulation stream's pressure to get outlet pressure.

        wi = ims.get_mass_flow() if ims.get_mass_flow() is not None else 0.0
        if self.flow_conductance > 0:
            pressure_drop_calculated = (wi / self.flow_conductance)**2
        else:
            pressure_drop_calculated = 0.0 # No pressure drop if no conductance

        ims.set_pressure(self.accumulation_stream.get_pressure()) # Inlet stream pressure set to accumulation pressure, seems odd

        oms.assign_from_phase("Mixture", self.accumulation_stream, False) # Assign properties to outlet stream
        oms.set_temperature(self.accumulation_stream.get_temperature())
        oms.set_mass_enthalpy(self.accumulation_stream.get_mass_enthalpy())
        oms.set_pressure(self.accumulation_stream.get_pressure() - pressure_drop_calculated)


    def calculate(self):
        """
        Performs the steady-state cooler calculations based on the selected mode.
        """
        # --- Pre-calculation checks and data retrieval ---
        ms_in = self.inlet_material_stream
        es_out = self.outlet_energy_stream

        if ms_in is None:
            raise Exception("Inlet material stream is not connected.")
        if es_out is None and self.calc_mode != self.CalculationMode.EnergyStream:
             # In DWSIM, if not in EnergyStream mode, it expects an energy outlet
             # This means the heat removed Q is an *output* of the cooler,
             # which is then assigned to an energy stream.
            pass # We'll handle assignment later if es_out is provided

        self.property_package.current_material_stream = ms_in

        ti = ms_in.get_temperature()
        pi = ms_in.get_pressure()
        hi = ms_in.get_mass_enthalpy()
        wi = ms_in.get_mass_flow()

        if None in [ti, pi, hi, wi]:
            raise Exception("Inlet stream properties (Temperature, Pressure, Enthalpy, Mass Flow) must be defined.")

        ei = hi * wi # Inlet energy flow (kW)

        # Outlet pressure calculation (P2 = P1 - DeltaP)
        p2 = pi - self.delta_p

        self.append_debug_line(f"Property Package: {self.property_package.__class__.__name__}")
        self.append_debug_line(f"Input variables: T = {ti} K, P = {pi} Pa, H = {hi} kJ/kg, W = {wi} kg/s")
        self.append_debug_line(f"Calculation mode: {self.calc_mode}")

        h2 = None # Outlet enthalpy
        t2 = None # Outlet temperature
        v2 = None # Outlet vapor fraction
        q_calc = None # Calculated heat duty

        # --- Main calculation logic based on CalculationMode ---
        if self.calc_mode == self.CalculationMode.EnergyStream:
            es_in = self.inlet_energy_stream
            if es_in is None:
                raise Exception("Secondary Energy Stream is required for EnergyStream calculation mode.")

            self.append_debug_line("Calculation Mode: Energy Stream")
            self.append_debug_line("Outlet Stream will be specified with Pressure and Enthalpy. Temperature will be calculated through a PH Flash call.")
            self.append_debug_line(f"H_2 = (Q / W) * (eta / 100) + H_1")

            self.delta_q = es_in.get_energy_flow()
            if wi == 0:
                raise Exception("Inlet mass flow rate cannot be zero for EnergyStream calculation.")
            h2 = (self.delta_q * (self.efficiency / 100)) / wi + hi

            self.append_debug_line(f"Q: {self.delta_q} kW, eta: {self.efficiency} %, W: {wi} kg/s, H_1: {hi} kJ/kg")
            self.append_debug_line(f"H_2: {h2} kJ/kg")

            self.append_debug_line(f"Doing a PH flash to calculate outlet temperature... P = {p2} Pa, H = {h2} kJ/kg")
            tmp_result = self.property_package.calculate_equilibrium2("PressureEnthalpy", p2, h2, ti)
            t2 = tmp_result.calculated_temperature
            self.check_spec(t2, True, "outlet temperature")
            self.delta_t = t2 - ti
            self.outlet_temperature = t2
            self.outlet_vapor_fraction = tmp_result.get_vapor_phase_mole_fraction()

            self.append_debug_line(f"Calculated outlet temperature T2 = {t2} K, Delta T = {self.delta_t} K")

        elif self.calc_mode == self.CalculationMode.HeatRemoved:
            self.append_debug_line("Calculation Mode: Heat Removed")
            self.append_debug_line("Outlet Stream will be specified with Pressure and Enthalpy. Temperature will be calculated through a PH Flash call.")
            self.append_debug_line(f"H_2 = (-Q / W) * (eta / 100) + H_1")

            q_input = self.delta_q
            if q_input is None:
                raise Exception("Heat Removed (DeltaQ) must be specified for HeatRemoved calculation mode.")

            if wi == 0:
                raise Exception("Inlet mass flow rate cannot be zero for HeatRemoved calculation.")

            h2 = (-q_input * (self.efficiency / 100)) / wi + hi
            self.check_spec(h2, False, "outlet enthalpy")

            self.append_debug_line(f"Q: {q_input} kW, eta: {self.efficiency} %, W: {wi} kg/s, H_1: {hi} kJ/kg")
            self.append_debug_line(f"H_2: {h2} kJ/kg")

            self.append_debug_line(f"Doing a PH flash to calculate outlet temperature... P = {p2} Pa, H = {h2} kJ/kg")
            tmp_result = self.property_package.calculate_equilibrium2("PressureEnthalpy", p2, h2, 0) # Use 0 as initial guess if not explicitly provided
            t2 = tmp_result.calculated_temperature
            self.check_spec(t2, True, "outlet temperature")
            self.delta_t = t2 - ti
            self.outlet_temperature = t2
            self.outlet_vapor_fraction = tmp_result.get_vapor_phase_mole_fraction()

            self.append_debug_line(f"Calculated outlet temperature T2 = {t2} K, Delta T = {self.delta_t} K")

            if es_out is not None:
                es_out.set_energy_flow(q_input) # Update energy stream with the input heat removed
            else:
                self.append_debug_line("Warning: No outlet energy stream connected to update heat duty.")


        elif self.calc_mode == self.CalculationMode.OutletTemperature:
            self.append_debug_line("Calculation Mode: Outlet Temperature")
            self.append_debug_line("Outlet Stream will be specified with Pressure and Temperature. Enthalpy will be calculated through a PT Flash call.")

            t2 = self.outlet_temperature
            if t2 is None:
                raise Exception("Outlet Temperature must be specified for OutletTemperature calculation mode.")

            self.append_debug_line(f"T_2: {t2} K, eta: {self.efficiency} %, W: {wi} kg/s, H_1: {hi} kJ/kg")

            self.append_debug_line(f"Doing a PT flash to calculate outlet enthalpy... P = {p2} Pa, T = {t2} K")
            tmp_result = self.property_package.calculate_equilibrium2("PressureTemperature", p2, t2, 0)
            h2 = tmp_result.calculated_enthalpy
            self.check_spec(h2, False, "outlet enthalpy")
            self.delta_t = t2 - ti
            q_calc = -(h2 - hi) / (self.efficiency / 100) * wi
            self.delta_q = q_calc
            self.outlet_vapor_fraction = tmp_result.get_vapor_phase_mole_fraction()

            self.append_debug_line(f"Delta T: {self.delta_t} K, H_2: {h2} kJ/kg")
            self.append_debug_line(f"Q = -(H_2 - H_1) / (eta / 100) * W")
            self.append_debug_line(f"Q: {q_calc} kW")

            if es_out is not None:
                es_out.set_energy_flow(q_calc)

        elif self.calc_mode == self.CalculationMode.TemperatureChange:
            self.append_debug_line("Calculation Mode: Temperature Change")
            self.append_debug_line("Outlet Stream will be specified with Pressure and Temperature. Enthalpy will be calculated through a PT Flash call.")

            delta_t_input = self.delta_t
            if delta_t_input is None:
                raise Exception("Temperature Change (DeltaT) must be specified for TemperatureChange calculation mode.")

            t2 = ti + delta_t_input
            self.outlet_temperature = t2 # Store the calculated outlet temperature

            self.append_debug_line(f"Delta T: {delta_t_input} K, eta: {self.efficiency} %, W: {wi} kg/s, H_1: {hi} kJ/kg")

            self.append_debug_line(f"Doing a PT flash to calculate outlet enthalpy... P = {p2} Pa, T = {t2} K")
            tmp_result = self.property_package.calculate_equilibrium2("PressureTemperature", p2, t2, 0)
            h2 = tmp_result.calculated_enthalpy
            self.check_spec(h2, False, "outlet enthalpy")
            q_calc = -(h2 - hi) / (self.efficiency / 100) * wi
            self.delta_q = q_calc
            self.outlet_vapor_fraction = tmp_result.get_vapor_phase_mole_fraction()

            self.append_debug_line(f"T_2: {t2} K, H_2: {h2} kJ/kg")
            self.append_debug_line(f"Q = -(H_2 - H_1) / (eta / 100) * W")
            self.append_debug_line(f"Q: {q_calc} kW")

            if es_out is not None:
                es_out.set_energy_flow(q_calc)

        elif self.calc_mode == self.CalculationMode.OutletVaporFraction:
            self.append_debug_line("Calculation Mode: Outlet Vapor Fraction")
            self.append_debug_line("Outlet Stream will be specified with Pressure and Temperature. Temperature will be calculated through a PVF Flash call.")

            v2 = self.outlet_vapor_fraction
            if v2 is None:
                raise Exception("Outlet Vapor Fraction must be specified for OutletVaporFraction calculation mode.")
            if not (0.0 <= v2 <= 1.0):
                raise Exception("Outlet Vapor Fraction must be between 0 and 1.")

            self.append_debug_line(f"VF_2: {v2}, eta: {self.efficiency} %, W: {wi} kg/s, H_1: {hi} kJ/kg")

            self.append_debug_line(f"Doing a PVF flash to calculate outlet temperature... P = {p2} Pa, VF = {v2}")
            tmp_result = self.property_package.calculate_equilibrium2("PressureVaporFraction", p2, v2, 0.0)
            h2 = tmp_result.calculated_enthalpy
            self.check_spec(h2, False, "outlet enthalpy")
            t2 = tmp_result.calculated_temperature
            self.check_spec(t2, True, "outlet temperature")
            self.delta_t = t2 - ti
            q_calc = -(h2 - hi) / (self.efficiency / 100) * wi
            self.delta_q = q_calc
            self.outlet_temperature = t2
            self.outlet_vapor_fraction = tmp_result.get_vapor_phase_mole_fraction() # Reconfirm the VF from flash

            self.append_debug_line(f"T_2: {t2} K, H_2: {h2} kJ/kg")
            self.append_debug_line(f"Q = -(H_2 - H_1) / (eta / 100) * W")
            self.append_debug_line(f"Q: {q_calc} kW")

            if es_out is not None:
                es_out.set_energy_flow(q_calc)

        else:
            raise Exception(f"Unsupported calculation mode: {self.calc_mode}")

        # --- Update outlet material stream with calculated values ---
        if self.outlet_material_stream is not None:
            self.outlet_material_stream.set_temperature(t2)
            self.outlet_material_stream.set_pressure(p2)
            self.outlet_material_stream.set_mass_enthalpy(h2)
            self.outlet_material_stream.set_vapor_phase_mole_fraction(v2) # Update vapor fraction as well
            # Copy components and mass flow from inlet to outlet (assuming no reaction/mass change)
            self.outlet_material_stream.components = ms_in.components.copy()
            self.outlet_material_stream.set_mass_flow(ms_in.get_mass_flow())
        else:
            self.append_debug_line("Warning: No outlet material stream connected to update properties.")

        self.append_debug_line("Cooler calculation finished successfully.")

    def get_average_molar_mass(self, components):
        """
        Placeholder for calculating average molar mass based on components and their mole fractions.
        Needs actual component molar masses.
        """
        # For a real implementation, you'd need a component database.
        # Let's assume a simple average molar mass if components are unknown
        if not components:
            return 0.029  # kg/mol, e.g., for air
        
        # Example for water, if it's the only component
        if "Water" in components and len(components) == 1:
            return 0.018015 # kg/mol for water
        
        # Fallback for unknown/multiple components
        return 0.029 # Default for approximation


# --- Example Usage ---
if __name__ == "__main__":
    # 1. Create a Property Package instance
    prop_pkg = PropertyPackage()

    # 2. Create Inlet Material Stream
    inlet_stream = MaterialStream(
        temperature=350.0,    # K
        pressure=200000.0,    # Pa (2 bar)
        enthalpy=1000.0,      # kJ/kg (example value)
        mass_flow=10.0,       # kg/s
        components={"Water": 1.0} # Pure water for simplicity
    )

    # 3. Create Outlet Material Stream
    outlet_stream = MaterialStream()

    # 4. Create an Energy Stream (for heat removed output or input)
    cooler_energy_stream = EnergyStream()

    # 5. Instantiate the Cooler
    cooler = Cooler("MyCooler")
    cooler.set_inlet_material_stream(inlet_stream)
    cooler.set_outlet_material_stream(outlet_stream)
    cooler.set_outlet_energy_stream(cooler_energy_stream) # Connect for output of Q
    cooler.set_property_package(prop_pkg)
    cooler.set_debug_mode(True)

    print("--- Calculating in Heat Removed Mode ---")
    cooler.calc_mode = Cooler.CalculationMode.HeatRemoved
    cooler.delta_p = 10000.0 # 10 kPa pressure drop
    cooler.delta_q = 5000.0  # 5000 kW heat removed (input)
    cooler.efficiency = 90.0 # 90% efficiency

    try:
        cooler.calculate()
        print(f"Cooler Calculation Mode: Heat Removed")
        print(f"Inlet T: {inlet_stream.get_temperature():.2f} K, P: {inlet_stream.get_pressure():.2f} Pa, H: {inlet_stream.get_mass_enthalpy():.2f} kJ/kg")
        print(f"Outlet T: {cooler.outlet_temperature:.2f} K, P: {outlet_stream.get_pressure():.2f} Pa, H: {outlet_stream.get_mass_enthalpy():.2f} kJ/kg")
        print(f"Heat Removed: {cooler.delta_q:.2f} kW")
        print(f"Delta T: {cooler.delta_t:.2f} K")
        print(f"Outlet Vapor Fraction: {cooler.outlet_vapor_fraction:.2f}")
        print("-" * 30)
    except Exception as e:
        print(f"Error in Heat Removed mode: {e}")
        print("-" * 30)

    # Reset outlet stream for next calculation
    outlet_stream = MaterialStream()
    cooler.set_outlet_material_stream(outlet_stream)

    print("\n--- Calculating in Outlet Temperature Mode ---")
    cooler.calc_mode = Cooler.CalculationMode.OutletTemperature
    cooler.delta_p = 10000.0
    cooler.outlet_temperature = 310.0 # Desired outlet temperature (input)
    cooler.efficiency = 95.0

    try:
        cooler.calculate()
        print(f"Cooler Calculation Mode: Outlet Temperature")
        print(f"Inlet T: {inlet_stream.get_temperature():.2f} K, P: {inlet_stream.get_pressure():.2f} Pa, H: {inlet_stream.get_mass_enthalpy():.2f} kJ/kg")
        print(f"Outlet T: {cooler.outlet_temperature:.2f} K, P: {outlet_stream.get_pressure():.2f} Pa, H: {outlet_stream.get_mass_enthalpy():.2f} kJ/kg")
        print(f"Heat Removed (calculated): {cooler.delta_q:.2f} kW")
        print(f"Delta T: {cooler.delta_t:.2f} K")
        print(f"Outlet Vapor Fraction: {cooler.outlet_vapor_fraction:.2f}")
        print("-" * 30)
    except Exception as e:
        print(f"Error in Outlet Temperature mode: {e}")
        print("-" * 30)

    # Reset for dynamic mode example
    outlet_stream = MaterialStream()
    cooler.set_outlet_material_stream(outlet_stream)
    cooler.delta_q = -1000.0 # Example heat duty for dynamic (negative for cooling)

    print("\n--- Running Dynamic Model Example ---")
    cooler.reset_content = True # Ensure accumulation stream is reset
    cooler.volume = 10.0 # m3
    cooler.flow_conductance = 0.001 # kg/s/Pa^0.5 (arbitrary value)

    # Simulate over several time steps
    print(f"Initial Inlet Mass Flow: {inlet_stream.get_mass_flow()} kg/s")
    for i in range(5):
        try:
            timestep = 1.0 # 1 second
            cooler.run_dynamic_model(timestep)
            print(f"Time Step {i+1}:")
            if cooler.accumulation_stream:
                print(f"  Accumulation Stream - T: {cooler.accumulation_stream.get_temperature():.2f} K, P: {cooler.accumulation_stream.get_pressure():.2f} Pa, H: {cooler.accumulation_stream.get_mass_enthalpy():.2f} kJ/kg, Mass Flow (acc): {cooler.accumulation_stream.get_mass_flow():.2f} kg/s")
            print(f"  Outlet Stream - T: {outlet_stream.get_temperature():.2f} K, P: {outlet_stream.get_pressure():.2f} Pa, H: {outlet_stream.get_mass_enthalpy():.2f} kJ/kg")
        except Exception as e:
            print(f"Error in dynamic model at step {i+1}: {e}")
        print("-" * 20)

In [4]:
import math
import csv

class IdealPropertyPackage:
    """
    A simple ideal property package for chemical process simulation.

    This class provides methods to calculate fundamental thermophysical properties
    based on ideal assumptions (Ideal Gas Law, Raoult's Law). It's a template
    that can be expanded with more complex models and real component data.

    Properties are now loaded from a CSV file, making it easy to add or modify
    component data without changing the code.
    """

    def __init__(self, components, data_file='ideal_properties.csv'):
        """
        Initializes the property package with a list of components and a data file.

        Args:
            components (list): A list of strings representing component names.
            data_file (str): The path to the CSV file containing property data.
        """
        self.components = components
        self.num_components = len(components)

        # Initialize dictionaries to store properties
        self.antoine_coeffs = {}
        self.cp_ideal_gas = {}
        self.molar_masses = {}
        self.liquid_molar_volumes = {}
        self.heats_of_vaporization = {}

        # Load property data from the specified CSV file
        self._load_properties_from_csv(data_file)

        self.R = 8.314  # Ideal gas constant (J/mol-K)

    def _load_properties_from_csv(self, file_path):
        """
        Loads thermophysical property data from a CSV file.

        The expected CSV format is:
        component,antoine_A,antoine_B,antoine_C,cp_ideal_gas,molar_mass,liquid_molar_volume,heat_of_vaporization
        
        Example CSV content:
        Component1,4.0,1000.0,-43.15,70.0,0.032,0.00005,30000.0
        Component2,4.5,1200.0,-50.0,80.0,0.046,0.00006,40000.0
        
        Args:
            file_path (str): The path to the CSV file.
        """
        try:
            with open(file_path, mode='r') as infile:
                reader = csv.DictReader(infile)
                for row in reader:
                    component = row['component']
                    if component in self.components:
                        # Convert string values to floats
                        self.antoine_coeffs[component] = {
                            'A': float(row['antoine_A']),
                            'B': float(row['antoine_B']),
                            'C': float(row['antoine_C'])
                        }
                        self.cp_ideal_gas[component] = float(row['cp_ideal_gas'])
                        self.molar_masses[component] = float(row['molar_mass'])
                        self.liquid_molar_volumes[component] = float(row['liquid_molar_volume'])
                        self.heats_of_vaporization[component] = float(row['heat_of_vaporization'])
        except FileNotFoundError:
            print(f"Error: The data file '{file_path}' was not found.")
            # For a real application, you might want to exit or use default values.
        except KeyError as e:
            print(f"Error: Missing column {e} in CSV file. Please check the header.")

    def _get_antoine_vapor_pressure(self, temp, component):
        """
        Calculates vapor pressure using the Antoine equation.
        Args:
            temp (float): Temperature in Kelvin.
            component (str): Component name.
        Returns:
            float: Vapor pressure in Pascals.
        """
        if component not in self.antoine_coeffs:
            raise ValueError(f"Antoine coefficients not found for {component}")

        coeffs = self.antoine_coeffs[component]
        A = coeffs['A']
        B = coeffs['B']
        C = coeffs['C']

        log_p = A - B / (temp + C)
        p_sat_bar = 10**log_p
        p_sat_pa = p_sat_bar * 100000
        return p_sat_pa

    def get_vapor_pressures(self, temp):
        """
        Calculates vapor pressure for all components at a given temperature.
        Args:
            temp (float): Temperature in Kelvin.
        Returns:
            dict: Dictionary with component names as keys and vapor pressures in Pa.
        """
        return {comp: self._get_antoine_vapor_pressure(temp, comp) for comp in self.components}

    def get_k_values(self, temp, pressure):
        """
        Calculates K-values for VLE using Raoult's Law (ideal solution).
        Args:
            temp (float): Temperature in Kelvin.
            pressure (float): Pressure in Pascals.
        Returns:
            dict: Dictionary with component names as keys and K-values.
        """
        vapor_pressures = self.get_vapor_pressures(temp)
        return {comp: vapor_pressures[comp] / pressure for comp in self.components}

    def get_enthalpy(self, temp, mole_fractions):
        """
        Calculates molar enthalpy relative to a reference temperature (298.15 K).
        Assumes ideal gas enthalpy and no enthalpy of mixing.
        Args:
            temp (float): Temperature in Kelvin.
            mole_fractions (dict): Component mole fractions.
        Returns:
            float: Molar enthalpy in J/mol.
        """
        ref_temp = 298.15
        total_enthalpy = 0
        for comp in self.components:
            if comp in self.cp_ideal_gas:
                cp = self.cp_ideal_gas[comp]
                enthalpy_change = cp * (temp - ref_temp)
                total_enthalpy += mole_fractions.get(comp, 0) * enthalpy_change
        return total_enthalpy

    def get_heat_capacity(self, mole_fractions):
        """
        Calculates the mixture's ideal gas heat capacity.
        Args:
            mole_fractions (dict): Component mole fractions.
        Returns:
            float: Molar heat capacity in J/mol-K.
        """
        mixture_cp = 0
        for comp in self.components:
            mixture_cp += mole_fractions.get(comp, 0) * self.cp_ideal_gas.get(comp, 0)
        return mixture_cp

    def get_specific_volume_gas(self, temp, pressure):
        """
        Calculates the specific volume of an ideal gas mixture.
        Args:
            temp (float): Temperature in Kelvin.
            pressure (float): Pressure in Pascals.
        Returns:
            float: Specific volume in m^3/mol.
        """
        return (self.R * temp) / pressure

    def get_molar_volume_liquid(self, mole_fractions):
        """
        Calculates the mixture's liquid molar volume (ideal solution).
        Args:
            mole_fractions (dict): Component mole fractions.
        Returns:
            float: Molar volume in m^3/mol.
        """
        total_molar_volume = 0
        for comp in self.components:
            total_molar_volume += mole_fractions.get(comp, 0) * self.liquid_molar_volumes.get(comp, 0)
        return total_molar_volume
    
    def get_heat_of_vaporization(self, component):
        """
        Retrieves the heat of vaporization for a component.
        Note: For simplicity, this is treated as a constant property from the CSV.
        In a more rigorous model, this would be a function of temperature.
        
        Args:
            component (str): The name of the component.
        Returns:
            float: The molar heat of vaporization in J/mol.
        """
        if component not in self.heats_of_vaporization:
            raise ValueError(f"Heat of vaporization not found for {component}")
        return self.heats_of_vaporization[component]

    def flash_calculation(self, temp, pressure, overall_mole_fractions, tolerance=1e-6, max_iterations=100):
        """
        Performs a flash calculation using the Rachford-Rice equation.

        Args:
            temp (float): Temperature in Kelvin.
            pressure (float): Pressure in Pascals.
            overall_mole_fractions (dict): Overall mole fractions of the feed stream.
            tolerance (float): The desired precision for the solution.
            max_iterations (int): Maximum number of iterations for the solver.

        Returns:
            tuple: (vapor_fraction, vapor_composition, liquid_composition) or None if no solution.
        """
        k_values = self.get_k_values(temp, pressure)

        # Check for all liquid or all vapor conditions
        sum_zk = sum(overall_mole_fractions.get(c, 0) * k_values.get(c, 0) for c in self.components)
        sum_z_over_k = sum(overall_mole_fractions.get(c, 0) / k_values.get(c, 0) for c in self.components)
        
        if sum_zk < 1:
            print("System is subcooled liquid (all liquid).")
            return 0.0, None, overall_mole_fractions
        if sum_z_over_k > 1:
            print("System is superheated vapor (all vapor).")
            return 1.0, overall_mole_fractions, None

        # Solve Rachford-Rice equation for vapor fraction (V/F)
        # We will use a simple iterative approach here
        def rachford_rice(vf):
            rr_sum = 0
            for comp in self.components:
                z = overall_mole_fractions.get(comp, 0)
                k = k_values.get(comp, 0)
                rr_sum += (z * (k - 1)) / (1 + vf * (k - 1))
            return rr_sum

        # Bisection method for root-finding
        vf_low = 0
        vf_high = 1
        vf = 0.5  # initial guess
        for i in range(max_iterations):
            f_low = rachford_rice(vf_low)
            f_high = rachford_rice(vf_high)
            f_mid = rachford_rice(vf)

            if abs(f_mid) < tolerance:
                break
            
            if f_mid * f_low < 0:
                vf_high = vf
            else:
                vf_low = vf
            
            vf = (vf_low + vf_high) / 2
        else:
            print("Warning: Flash calculation did not converge.")

        # Calculate compositions
        liquid_composition = {}
        vapor_composition = {}
        
        for comp in self.components:
            z = overall_mole_fractions.get(comp, 0)
            k = k_values.get(comp, 0)
            x_i = z / (1 + vf * (k - 1))
            y_i = k * x_i
            liquid_composition[comp] = x_i
            vapor_composition[comp] = y_i

        return vf, vapor_composition, liquid_composition

# --- Example Usage ---
if __name__ == "__main__":
    # You must have a file named 'ideal_properties.csv' in the same directory.
    # For this example, let's pretend we have it.
    components_list = ['Component1', 'Component2']
    pp = IdealPropertyPackage(components_list, 'ideal_properties.csv')

    # Define a state (T, P, composition)
    T = 350.0 # Kelvin
    P = 150000.0 # Pascals (1.5 bar)
    overall_moles = {'Component1': 0.5, 'Component2': 0.5}

    # Perform flash calculation
    vf, vapor_comp, liquid_comp = pp.flash_calculation(T, P, overall_moles)

    # Print the results
    if vf is not None:
      print(f"\nFlash Calculation Results at {T} K and {P/100000:.1f} bar:")
      print(f"Vapor Fraction (V/F): {vf:.4f}")
      if vapor_comp is not None:
          print("Vapor Composition (y):")
          for comp, mole_frac in vapor_comp.items():
              print(f"  {comp}: {mole_frac:.4f}")
      if liquid_comp is not None:
          print("Liquid Composition (x):")
          for comp, mole_frac in liquid_comp.items():
              print(f"  {comp}: {mole_frac:.4f}")



Error: The data file 'ideal_properties.csv' was not found.


ValueError: Antoine coefficients not found for Component1

In [ ]:
flow_stream

In [ ]:
# heater

import math

class MaterialStream:
    """
    A simplified class to represent a material stream in a chemical process.
    It holds properties like temperature, pressure, molar flow, composition,
    and now includes a calculated enthalpy rate.
    
    In a real simulator like DWSIM, this would be linked to a
    thermodynamic package for property calculations.
    """
    def __init__(self, temperature, pressure, molar_flow, components):
        """
        Initializes the material stream.

        :param temperature: Temperature in Kelvin (K).
        :param pressure: Pressure in Pascal (Pa).
        :param molar_flow: Total molar flow rate in mol/s.
        :param components: A dictionary of component mole fractions, e.g.,
                           {'methane': 0.8, 'ethane': 0.2}.
                           The sum of mole fractions must be 1.0.
        """
        self.temperature = temperature
        self.pressure = pressure
        self.molar_flow = molar_flow
        
        # Ensure mole fractions sum to 1.0
        if not math.isclose(sum(components.values()), 1.0):
            raise ValueError("Component mole fractions must sum to 1.0")
        self.components = components
        
        # A very simple, fixed heat capacity for demonstration purposes.
        # This would be a complex, temperature-dependent function in a real simulator.
        self.heat_capacity = self._calculate_mixture_heat_capacity()

        # Calculate the enthalpy rate of the stream.
        # This is a new attribute to fulfill your request.
        self.enthalpy_rate = self._calculate_enthalpy_rate()

    def _calculate_mixture_heat_capacity(self):
        """
        Calculates a simplified mixture heat capacity (Cp) in J/(mol*K).
        This is a placeholder and assumes a fixed Cp for each component.
        A real model would use a thermodynamic package.
        """
        # Fixed heat capacities (in J/mol*K) for example components.
        # These are just illustrative values.
        component_cp = {
            'methane': 35.7,
            'ethane': 52.6,
            'propane': 73.6
        }
        
        mixture_cp = 0.0
        for comp, mole_fraction in self.components.items():
            if comp in component_cp:
                mixture_cp += mole_fraction * component_cp[comp]
            else:
                print(f"Warning: Heat capacity for '{comp}' not found. Assuming 0.")
        return mixture_cp

    def _calculate_enthalpy_rate(self):
        """
        Calculates the enthalpy rate of the stream in Watts (J/s).
        
        This is a simplified calculation based on a reference state.
        Enthalpy is calculated as:
        $\dot{H} = \dot{n} \cdot C_p \cdot (T - T_{ref}) + \dot{H}_{pressure\_correction}$
        
        For this simplified model, we ignore the pressure correction and
        assume a reference temperature ($T_{ref}$) of 298.15 K (25°C).
        """
        # Define a reference temperature for enthalpy calculation
        T_ref = 298.15  # Kelvin
        
        # Calculate the enthalpy per mole relative to the reference temperature
        enthalpy_per_mole = self.heat_capacity * (self.temperature - T_ref)
        
        # Calculate the total enthalpy rate
        total_enthalpy_rate = self.molar_flow * enthalpy_per_mole
        
        return total_enthalpy_rate

    def __repr__(self):
        """
        Provides a string representation of the stream for printing.
        """
        return (f"MaterialStream(T={self.temperature:.2f} K, P={self.pressure / 1000:.2f} kPa, "
                f"Flow={self.molar_flow:.2f} mol/s, Cp_mix={self.heat_capacity:.2f} J/mol·K, "
                f"H_rate={self.enthalpy_rate:.2f} W)")

class Heater:
    """
    A simplified class to represent a heater unit operation.
    It calculates the outlet stream's temperature and enthalpy rate based
    on a given heat duty, assuming no phase change and constant heat capacity.
    """
    def __init__(self, name="Heater"):
        """
        Initializes the heater.
        """
        self.name = name
        self.inlet_stream = None
        self.outlet_stream = None
        
    def run_simulation(self, inlet_stream, heat_duty, outlet_pressure=None):
        """
        Runs the heater simulation to calculate the outlet stream properties.
        This simplified version assumes constant pressure and no phase change.

        The fundamental equation used is:
        $Q = \Delta \dot{H} = \dot{H}_{outlet} - \dot{H}_{inlet}$
        and
        $Q = \dot{n} \cdot C_p \cdot \Delta T$
        
        :param inlet_stream: The MaterialStream object entering the heater.
        :param heat_duty: The amount of heat added to the stream in Watts (J/s).
        :param outlet_pressure: The pressure of the outlet stream in Pascal (Pa).
                                If None, outlet pressure equals inlet pressure.
        :return: A new MaterialStream object representing the outlet stream.
        """
        if not isinstance(inlet_stream, MaterialStream):
            raise TypeError("Inlet must be a MaterialStream object.")
        
        self.inlet_stream = inlet_stream
        
        # Use inlet pressure if no outlet pressure is specified.
        out_pressure = outlet_pressure if outlet_pressure is not None else inlet_stream.pressure
        
        # Check for potential errors like zero flow
        if inlet_stream.molar_flow <= 0:
            print("Warning: Zero or negative molar flow rate. Outlet stream will be identical to inlet.")
            self.outlet_stream = MaterialStream(
                inlet_stream.temperature, out_pressure, inlet_stream.molar_flow, inlet_stream.components
            )
            return self.outlet_stream

        # Calculate the temperature change using the energy balance equation.
        # Delta T = Q / (n * Cp)
        delta_t = heat_duty / (inlet_stream.molar_flow * inlet_stream.heat_capacity)
        
        outlet_temperature = inlet_stream.temperature + delta_t
        
        # Create and return the new outlet stream
        self.outlet_stream = MaterialStream(
            outlet_temperature, out_pressure, inlet_stream.molar_flow, inlet_stream.components
        )
        return self.outlet_stream

# --- Example Usage ---
if __name__ == "__main__":
    try:
        # Define the inlet stream properties
        inlet_props = {
            'temperature': 300,  # K
            'pressure': 101325,  # Pa (1 atm)
            'molar_flow': 10,    # mol/s
            'components': {'methane': 0.8, 'ethane': 0.2}
        }
        
        # Create the inlet stream object
        inlet_stream_1 = MaterialStream(**inlet_props)
        print("Inlet Stream:")
        print(inlet_stream_1)
        print("-" * 20)
        
        # Create a heater object
        my_heater = Heater(name="Example Heater 1")
        
        # Define the heat duty to be added
        heat_duty_1 = 50000  # Watts (J/s)
        
        # Run the simulation
        outlet_stream_1 = my_heater.run_simulation(inlet_stream_1, heat_duty_1)
        
        # Print the results
        print(f"Simulation of '{my_heater.name}' with heat duty of {heat_duty_1} W:")
        print("Outlet Stream:")
        print(outlet_stream_1)
        
        # --- Another example with a different heat duty ---
        print("\n" + "=" * 30 + "\n")
        
        # Create another heater object
        my_heater_2 = Heater(name="Example Heater 2")
        
        # Define a larger heat duty
        heat_duty_2 = 100000  # Watts (J/s)
        
        # Run the simulation again with the same inlet stream
        outlet_stream_2 = my_heater_2.run_simulation(inlet_stream_1, heat_duty_2)
        
        # Print the results
        print(f"Simulation of '{my_heater_2.name}' with heat duty of {heat_duty_2} W:")
        print("Outlet Stream:")
        print(outlet_stream_2)

    except ValueError as e:
        print(f"Error: {e}")
    except TypeError as e:
        print(f"Error: {e}")
